In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, \
    f1_score, roc_curve, auc, precision_recall_curve, average_precision_score, roc_auc_score
from xgboost import XGBClassifier
from IPython.display import display
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

HERE = Path.cwd()
if (HERE / "main.py").exists():
    REPO = HERE.parent
    MODELS_DIR = HERE / "models"
else:
    REPO = HERE
    MODELS_DIR = HERE / "logistic_regression" / "models"
TELCO_CSV = REPO / "data" / "telco" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:


# Load the dataset
data = pd.read_csv(TELCO_CSV)

# Remove leading and trailing spaces in all columns
data = data.apply(lambda x: x.str.strip() if x.dtype == "object" else x)


In [ ]:
data

In [ ]:
data.shape

In [ ]:
data.size


In [ ]:
scalar_value = data.size

In [ ]:
scalar_value

## Exploratory Data Analysis (EDA) & distributions

Explore class balance, feature distributions, and how variables relate to **Churn** before modeling.  
After each code cell, a short note explains **what the plot/table means using real values from this dataset**.


### Target variable: is Churn balanced?


In [ ]:
churn_counts = data["Churn"].value_counts()
churn_pct = data["Churn"].value_counts(normalize=True).mul(100).round(1)

summary = pd.DataFrame({"count": churn_counts, "percent": churn_pct})
print(summary)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
churn_counts.plot(kind="bar", ax=axes[0], color=["steelblue", "salmon"], rot=0)
axes[0].set_title("Churn counts")
axes[0].set_xlabel("Churn")
axes[0].set_ylabel("Customers")

axes[1].pie(
    churn_counts,
    labels=churn_counts.index,
    autopct="%1.1f%%",
    colors=["steelblue", "salmon"],
    startangle=90,
)
axes[1].set_title("Churn share")
plt.tight_layout()
plt.show()


#### Plot explanation — Churn balance / توضیح نمودار — تعادل ریزش مشتری

**English**

- **Left — bar chart:** number of customers who stayed vs left. **No = 5,174**, **Yes = 1,869**.
- **Right — pie chart:** same as percentages — stayed **~73.5%**, churned **~26.5%**.
- **Meaning:** most customers do **not** churn. The target is imbalanced, so accuracy alone can be misleading.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

- **نمودار میله‌ای (سمت چپ):** تعداد مشتریانی که مانده‌اند در برابر کسانی که رفته‌اند.
  - مانده‌اند (No): **۵۱۷۴ نفر**
  - ریزش کرده‌اند (Yes): **۱۸۶۹ نفر**
- **نمودار دایره‌ای (سمت راست):** همان اطلاعات به صورت درصد:
  - حدود **۷۳٫۵٪** مانده‌اند
  - حدود **۲۶٫۵٪** ریزش کرده‌اند
- **نتیجه:** بیشتر مشتریان ریزش نمی‌کنند. چون داده‌ها نامتوازن است، فقط به دقت مدل (Accuracy) بسنده نکنید.

</div>


### Numeric feature distributions

Histograms and density curves for `tenure`, `MonthlyCharges`, and `TotalCharges`.


In [ ]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, num_cols):
    data[col].dropna().hist(bins=30, ax=ax, color="steelblue", edgecolor="white")
    ax.set_title(f"Distribution of {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

data[num_cols].describe().round(2)


#### Plot explanation — Histograms / توضیح نمودار — هیستوگرام ویژگی‌های عددی

**English**

- **Left — `tenure`:** mean **32.4**, median **29**, max **72**. Many customers are new; another group stays near 72 months.
- **Middle — `MonthlyCharges`:** mean **$64.76**, median **$70.35**, roughly **$18–$119**. Spike at cheaper plans, wide spread at higher bills.
- **Right — `TotalCharges`:** mean **~$2,283**, median **~$1,397**, max **~$8,685**. Right-skewed (long tail of high spenders).

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

هر ستون نشان می‌دهد چند مشتری در آن بازه قرار دارند.

- **سمت چپ — مدت اشتراک (`tenure`):**
  - میانگین **۳۲٫۴** ماه، میانه **۲۹** ماه، حداکثر **۷۲** ماه
  - بسیاری مشتری تازه‌وارد هستند و گروهی هم نزدیک ۷۲ ماه می‌مانند
- **وسط — هزینه ماهانه (`MonthlyCharges`):**
  - میانگین **۶۴٫۷۶ دلار**، میانه **۷۰٫۳۵ دلار**
  - بازه تقریبی از **۱۸ تا ۱۱۹ دلار**
  - اوج در پلن‌های ارزان و پراکندگی زیاد در صورتحساب‌های بالاتر
- **سمت راست — مجموع هزینه‌ها (`TotalCharges`):**
  - میانگین حدود **۲۲۸۳ دلار**، میانه حدود **۱۳۹۷ دلار**، حداکثر حدود **۸۶۸۵ دلار**
  - شکل نمودار به راست کشیده است (چند مشتری پرمصرف میانگین را بالا می‌کشند)

</div>


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, num_cols):
    sns.kdeplot(data=data, x=col, fill=True, ax=ax, color="teal")
    ax.set_title(f"Density of {col}")
plt.tight_layout()
plt.show()


#### Plot explanation — Density (KDE) / توضیح نمودار — منحنی چگالی

**English**

A KDE is a smooth version of the histogram (higher curve = more customers near that value).
- **tenure:** higher density at short tenure and again toward long tenure.
- **MonthlyCharges:** density rises near **~$70+** (matches median **$70.35**).
- **TotalCharges:** peak at lower totals, then a long right tail — median (**~$1,397**) below mean (**~$2,283**).

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

منحنی KDE نسخهٔ نرم هیستوگرام است. هر جا منحنی بالاتر باشد، مشتریان بیشتری در آن مقدار هستند.

- **مدت اشتراک:** تراکم در ماه‌های کم زیاد است و دوباره در ماه‌های زیاد بالا می‌رود.
- **هزینه ماهانه:** تراکم اطراف **۷۰ دلار به بالا** بیشتر است (هم‌خوان با میانه **۷۰٫۳۵**).
- **مجموع هزینه‌ها:** اوج در مقادیر پایین است و بعد یک دم بلند به سمت راست دارد؛ به همین دلیل میانه (**حدود ۱۳۹۷**) از میانگین (**حدود ۲۲۸۳**) کمتر است.

</div>


### Numeric features by Churn


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, num_cols):
    sns.histplot(
        data=data,
        x=col,
        hue="Churn",
        bins=30,
        element="step",
        stat="density",
        common_norm=False,
        ax=ax,
    )
    ax.set_title(f"{col} by Churn")
plt.tight_layout()
plt.show()


#### Plot explanation — Features by Churn / توضیح نمودار — ویژگی‌های عددی بر اساس ریزش

**English**

Two colors = Churn Yes vs No (densities so 1,869 churners stay visible next to 5,174 stayers).
- **tenure:** churners cluster at low tenure (**~18 mo**) vs stayers (**~38 mo**).
- **MonthlyCharges:** churners higher bills (**~$74**) vs stayers (**~$61**).
- **TotalCharges:** churners often lower totals (**~$1,532**) because they leave earlier; stayers **~$2,555**.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

دو رنگ یعنی مقایسهٔ مشتریان ریزش‌کرده با مشتریان مانده. از چگالی استفاده شده تا گروه کوچک‌تر (۱۸۶۹ نفر) کنار گروه بزرگ‌تر (۵۱۷۴ نفر) دیده شود.

- **مدت اشتراک:** ریزش‌کرده‌ها بیشتر در ماه‌های کم هستند (**حدود ۱۸ ماه**)؛ مانده‌ها حدود **۳۸ ماه**.
- **هزینه ماهانه:** ریزش‌کرده‌ها صورتحساب بالاتری دارند (**حدود ۷۴ دلار**) نسبت به مانده‌ها (**حدود ۶۱ دلار**).
- **مجموع هزینه‌ها:** ریزش‌کرده‌ها معمولاً مجموع کمتری دارند (**حدود ۱۵۳۲ دلار**) چون زودتر می‌روند؛ مانده‌ها حدود **۲۵۵۵ دلار**.

</div>


### Categorical feature distributions

Each category is plotted in its **own cell** below, with an English + فارسی explanation under every plot.


In [ ]:
# Category columns we will plot one-by-one
cat_cols = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "InternetService",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
]
print(cat_cols)


In [ ]:
# Count plot: gender
order = data["gender"].value_counts().index
plt.figure(figsize=(7, 4))
sns.countplot(data=data, x="gender", order=order, color="steelblue")
plt.title("Count of customers by gender")
plt.xlabel("gender")
plt.ylabel("Number of customers")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print(data["gender"].value_counts())


#### Plot — gender counts / نمودار — تعداد بر اساس جنسیت

**English**

This bar chart shows how many customers are Male vs Female.
- **Male: 3,555**
- **Female: 3,488**

**Meaning:** gender is almost perfectly balanced in this dataset, so gender alone is unlikely to drive big churn differences.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

این نمودار میله‌ای تعداد مشتریان مرد و زن را نشان می‌دهد.
- **مرد: ۳۵۵۵ نفر**
- **زن: ۳۴۸۸ نفر**

**معنی:** توزیع جنسیت تقریباً برابر است؛ بنابراین جنسیت به‌تنهایی عامل قوی ریزش به نظر نمی‌رسد.

</div>


In [ ]:
# Count plot: Partner
order = data["Partner"].value_counts().index
plt.figure(figsize=(7, 4))
sns.countplot(data=data, x="Partner", order=order, color="steelblue")
plt.title("Count of customers by Partner")
plt.xlabel("Partner")
plt.ylabel("Number of customers")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print(data["Partner"].value_counts())


#### Plot — Partner counts / نمودار — تعداد بر اساس داشتن شریک

**English**

Compares customers with and without a partner.
- **No partner: 3,641**
- **Has partner: 3,402**

Slightly more customers have no partner. Later churn-rate plots show no-partner customers churn more often.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

مقایسه مشتریان با شریک و بدون شریک.
- **بدون شریک: ۳۶۴۱ نفر**
- **با شریک: ۳۴۰۲ نفر**

کمی بیشتر بدون شریک هستند. در نمودارهای درصد ریزش می‌بینیم که گروه بدون شریک معمولاً ریزش بیشتری دارد.

</div>


In [ ]:
# Count plot: Dependents
order = data["Dependents"].value_counts().index
plt.figure(figsize=(7, 4))
sns.countplot(data=data, x="Dependents", order=order, color="steelblue")
plt.title("Count of customers by Dependents")
plt.xlabel("Dependents")
plt.ylabel("Number of customers")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print(data["Dependents"].value_counts())


#### Plot — Dependents counts / نمودار — تعداد بر اساس افراد تحت تکفل

**English**

- **No dependents: 4,933**
- **Has dependents: 2,110**

Most customers do **not** have dependents. This is an imbalanced category feature.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

- **بدون تحت‌تکفل: ۴۹۳۳ نفر**
- **دارای تحت‌تکفل: ۲۱۱۰ نفر**

بیشتر مشتریان افراد تحت تکفل ندارند. این ویژگی از نظر تعداد نامتوازن است.

</div>


In [ ]:
# Count plot: PhoneService
order = data["PhoneService"].value_counts().index
plt.figure(figsize=(7, 4))
sns.countplot(data=data, x="PhoneService", order=order, color="steelblue")
plt.title("Count of customers by PhoneService")
plt.xlabel("PhoneService")
plt.ylabel("Number of customers")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print(data["PhoneService"].value_counts())


#### Plot — PhoneService counts / نمودار — تعداد بر اساس سرویس تلفن

**English**

- **Yes (has phone): 6,361**
- **No: 682**

Almost all customers have phone service. The “No” group is small, so compare carefully when looking at churn rates.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

- **بله (دارای تلفن): ۶۳۶۱ نفر**
- **خیر: ۶۸۲ نفر**

تقریباً همه مشتریان سرویس تلفن دارند. گروه «خیر» کوچک است؛ پس در مقایسه درصد ریزش با احتیاط تفسیر کنید.

</div>


In [ ]:
# Count plot: InternetService
order = data["InternetService"].value_counts().index
plt.figure(figsize=(7, 4))
sns.countplot(data=data, x="InternetService", order=order, color="steelblue")
plt.title("Count of customers by InternetService")
plt.xlabel("InternetService")
plt.ylabel("Number of customers")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print(data["InternetService"].value_counts())


#### Plot — InternetService counts / نمودار — تعداد بر اساس نوع اینترنت

**English**

- **Fiber optic: 3,096**
- **DSL: 2,421**
- **No internet: 1,526**

Fiber is the largest group. Later we see Fiber also has a much higher churn rate than DSL or no internet.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

- **فیبر نوری: ۳۰۹۶ نفر**
- **DSL: ۲۴۲۱ نفر**
- **بدون اینترنت: ۱۵۲۶ نفر**

فیبر بزرگ‌ترین گروه است. در ادامه می‌بینیم که نرخ ریزش فیبر هم خیلی بالاتر از DSL یا بدون اینترنت است.

</div>


In [ ]:
# Count plot: Contract
order = data["Contract"].value_counts().index
plt.figure(figsize=(7, 4))
sns.countplot(data=data, x="Contract", order=order, color="steelblue")
plt.title("Count of customers by Contract")
plt.xlabel("Contract")
plt.ylabel("Number of customers")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print(data["Contract"].value_counts())


#### Plot — Contract counts / نمودار — تعداد بر اساس نوع قرارداد

**English**

- **Month-to-month: 3,875** (largest)
- **Two year: 1,695**
- **One year: 1,473**

Most customers are on flexible month-to-month contracts — the same group that later shows the highest churn (~42.7%).

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

- **ماهانه: ۳۸۷۵ نفر** (بیشترین)
- **دوساله: ۱۶۹۵ نفر**
- **یک‌ساله: ۱۴۷۳ نفر**

بیشتر مشتریان قرارداد ماهانه دارند؛ همان گروهی که بعداً بالاترین ریزش را نشان می‌دهد (حدود ۴۲٫۷٪).

</div>


In [ ]:
# Count plot: PaperlessBilling
order = data["PaperlessBilling"].value_counts().index
plt.figure(figsize=(7, 4))
sns.countplot(data=data, x="PaperlessBilling", order=order, color="steelblue")
plt.title("Count of customers by PaperlessBilling")
plt.xlabel("PaperlessBilling")
plt.ylabel("Number of customers")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print(data["PaperlessBilling"].value_counts())


#### Plot — PaperlessBilling counts / نمودار — تعداد بر اساس صورتحساب بدون کاغذ

**English**

- **Yes: 4,171**
- **No: 2,872**

More customers use paperless billing. This feature often relates to digital/self-serve behavior and can link to churn patterns.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

- **بله: ۴۱۷۱ نفر**
- **خیر: ۲۸۷۲ نفر**

بیشتر مشتریان صورتحساب بدون کاغذ دارند. این ویژگی معمولاً با رفتار دیجیتال مرتبط است و می‌تواند با الگوی ریزش پیوند داشته باشد.

</div>


In [ ]:
# Count plot: PaymentMethod
order = data["PaymentMethod"].value_counts().index
plt.figure(figsize=(7, 4))
sns.countplot(data=data, x="PaymentMethod", order=order, color="steelblue")
plt.title("Count of customers by PaymentMethod")
plt.xlabel("PaymentMethod")
plt.ylabel("Number of customers")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print(data["PaymentMethod"].value_counts())


#### Plot — PaymentMethod counts / نمودار — تعداد بر اساس روش پرداخت

**English**

- **Electronic check: 2,365** (largest)
- **Mailed check: 1,612**
- **Bank transfer (automatic): 1,544**
- **Credit card (automatic): 1,522**

Electronic check is most common and later has the highest churn rate (~45.3%).

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

- **چک الکترونیک: ۲۳۶۵ نفر** (بیشترین)
- **چک پستی: ۱۶۱۲ نفر**
- **انتقال بانکی خودکار: ۱۵۴۴ نفر**
- **کارت اعتباری خودکار: ۱۵۲۲ نفر**

چک الکترونیک رایج‌ترین روش است و بعداً بالاترین نرخ ریزش را دارد (حدود ۴۵٫۳٪).

</div>


### Churn rate by category


In [ ]:
def churn_rate_table(column):
    tab = pd.crosstab(data[column], data["Churn"], normalize="index").mul(100).round(1)
    tab["n"] = data[column].value_counts()
    return tab.sort_values("Yes", ascending=False)

for col in ["Contract", "InternetService", "PaymentMethod", "Partner", "SeniorCitizen"]:
    print(f"\n=== Churn % by {col} ===")
    print(churn_rate_table(col))


In [ ]:
plot_cols = ["Contract", "InternetService", "PaymentMethod", "TechSupport"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

for ax, col in zip(axes, plot_cols):
    rates = (
        data.groupby(col)["Churn"]
        .apply(lambda s: (s == "Yes").mean() * 100)
        .sort_values(ascending=False)
    )
    rates.plot(kind="bar", ax=ax, color="salmon", rot=30)
    ax.set_title(f"Churn rate (%) by {col}")
    ax.set_ylabel("Churn %")
    ax.set_xlabel("")
    ax.set_ylim(0, max(55, rates.max() + 5))

plt.tight_layout()
plt.show()


#### Plot explanation — Churn rate by category / توضیح نمودار — درصد ریزش در هر دسته

**English**

Y-axis = % who churned (overall baseline **~26.5%**). Taller bar = riskier group.
- **Contract:** Month-to-month **~42.7%**; One year **~11.3%**; Two year **~2.8%**.
- **InternetService:** Fiber **~41.9%**; DSL **~19.0%**; No internet **~7.4%**.
- **PaymentMethod:** Electronic check **~45.3%** (highest); auto card/bank ~**15–17%**.
- **TechSupport:** No **~41.6%**; Yes **~15.2%**; no internet **~7.4%**.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

محور عمودی = درصد کسانی که در آن گروه ریزش کرده‌اند.  
پایه کل داده‌ها حدود **۲۶٫۵٪** است. میله بلندتر یعنی گروه پرریسک‌تر.

- **قرارداد:** ماهانه حدود **۴۲٫۷٪**؛ یک‌ساله **۱۱٫۳٪**؛ دوساله فقط **۲٫۸٪**
- **اینترنت:** فیبر حدود **۴۱٫۹٪**؛ DSL حدود **۱۹٪**؛ بدون اینترنت حدود **۷٫۴٪**
- **روش پرداخت:** چک الکترونیک حدود **۴۵٫۳٪** (بیشترین ریسک)؛ کارت یا بانک خودکار حدود **۱۵ تا ۱۷٪**
- **پشتیبانی فنی:** بدون پشتیبانی حدود **۴۱٫۶٪**؛ با پشتیبانی حدود **۱۵٫۲٪**؛ بدون اینترنت حدود **۷٫۴٪**

**نتیجه ساده:** قرارداد ماهانه، اینترنت فیبر، چک الکترونیک و نداشتن پشتیبانی فنی با ریزش بالاتر همراه هستند.

</div>


### Correlation between numeric features / همبستگی بین ویژگی‌های عددی

**English**

This section checks how numeric columns move together, and how each one relates to churn.
We create `ChurnFlag` (Yes→1, No→0) so churn can be included in the correlation table and heatmap.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

در این بخش می‌بینیم ویژگی‌های عددی چقدر با هم حرکت می‌کنند و هر کدام چه ربطی به ریزش دارد.
ستون `ChurnFlag` ساخته می‌شود (بله→۱، خیر→۰) تا ریزش هم داخل جدول و نقشه همبستگی بیاید.

</div>


In [ ]:
data["ChurnFlag"] = data["Churn"].map({"Yes": 1, "No": 0})

corr_cols = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen", "ChurnFlag"]
corr = data[corr_cols].corr().round(2)
print(corr)

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, vmin=-1, vmax=1, square=True)
plt.title("Correlation heatmap")
plt.tight_layout()
plt.show()


#### What this correlation heatmap shows / این نقشه همبستگی چه می‌گوید

**English**

Each cell is a correlation from **-1 to +1** (red = positive, blue = negative).

Important values in this data:
- **tenure ↔ TotalCharges ≈ 0.83:** longer customers usually paid more in total.
- **MonthlyCharges ↔ TotalCharges ≈ 0.65:** higher monthly bills raise total spend.
- **tenure ↔ ChurnFlag ≈ -0.35:** longer tenure is linked to **less** churn.
- **MonthlyCharges ↔ ChurnFlag ≈ 0.19:** higher monthly bill is linked to **slightly more** churn.
- **TotalCharges ↔ ChurnFlag ≈ -0.20:** higher totals link to somewhat less churn (often because stayers stayed longer).
- **SeniorCitizen ↔ ChurnFlag ≈ 0.15:** seniors are slightly more associated with churn.

**Remember:** correlation is association, not proof of cause.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

هر خانه یک عدد همبستگی بین **-۱ و +۱** است (قرمز = مثبت، آبی = منفی).

مقادیر مهم در این داده:
- **مدت اشتراک و مجموع هزینه ≈ ۰٫۸۳:** مشتریانی که بیشتر مانده‌اند معمولاً مجموع بیشتری پرداخته‌اند.
- **هزینه ماهانه و مجموع ≈ ۰٫۶۵:** صورتحساب ماهانه بالاتر، مجموع را هم بالا می‌برد.
- **مدت و ریزش ≈ -۰٫۳۵:** مدت بیشتر با ریزش کمتر مرتبط است.
- **هزینه ماهانه و ریزش ≈ ۰٫۱۹:** صورتحساب بالاتر کمی با ریزش بیشتر مرتبط است.
- **مجموع و ریزش ≈ -۰٫۲۰:** مجموع بالاتر کمی با ریزش کمتر مرتبط است (اغلب چون مانده‌ها بیشتر مانده‌اند).
- **سالمند و ریزش ≈ ۰٫۱۵:** سالمندان کمی بیشتر با ریزش مرتبط‌اند.

**یادآوری:** همبستگی فقط ارتباط را نشان می‌دهد، نه علت قطعی را.

</div>


### Scatter: tenure vs charges / نمودار پراکندگی: مدت در برابر هزینه

**English**

Scatter plots show each customer as one point. Color = churn status.
We sample up to **1,000** rows so the plot stays readable.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

در نمودار پراکندگی هر نقطه یک مشتری است. رنگ = وضعیت ریزش.
تا **۱۰۰۰** ردیف نمونه‌گیری می‌کنیم تا نمودار خوانا بماند.

</div>


In [ ]:
sample = data.sample(n=min(1000, len(data)), random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(
    data=sample, x="tenure", y="MonthlyCharges", hue="Churn", alpha=0.5, ax=axes[0]
)
axes[0].set_title("Tenure vs MonthlyCharges")

sns.scatterplot(
    data=sample, x="tenure", y="TotalCharges", hue="Churn", alpha=0.5, ax=axes[1]
)
axes[1].set_title("Tenure vs TotalCharges")

plt.tight_layout()
plt.show()


#### How to read these two scatter plots / چطور این دو نمودار پراکندگی را بخوانیم

**English**

- **Left (tenure vs MonthlyCharges):** churned customers appear more at **low tenure** and often **higher monthly charges** (about **18 months / $74** vs **38 months / $61**).
- **Right (tenure vs TotalCharges):** clear upward trend (correlation **~0.83**). Churners cluster toward the bottom-left (short time, smaller totals).
- **Key idea:** low total charges among churners often means they left early, not only that they had cheap plans.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

- **سمت چپ (مدت در برابر هزینه ماهانه):** ریزش‌کرده‌ها بیشتر در **مدت کم** و اغلب با **صورتحساب ماهانه بالاتر** دیده می‌شوند (حدود **۱۸ ماه / ۷۴ دلار** در برابر **۳۸ ماه / ۶۱ دلار**).
- **سمت راست (مدت در برابر مجموع):** روند صعودی واضح است (همبستگی حدود **۰٫۸۳**). نقاط ریزش بیشتر در پایین-چپ جمع شده‌اند.
- **نکته کلیدی:** اگر مجموع هزینه ریزش‌کرده‌ها کم است، اغلب به‌خاطر ترک زودهنگام است نه فقط پلن ارزان.

</div>


### EDA takeaways / جمع‌بندی تحلیل اکتشافی

**English**

This cell prints the strongest EDA conclusions in one place before modeling.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

این سلول مهم‌ترین نتیجه‌های تحلیل اکتشافی را قبل از مدل‌سازی یکجا چاپ می‌کند.

</div>


In [ ]:
print("Dataset:", data.shape[0], "customers,", data.shape[1], "columns")
print("Overall churn rate: {:.1f}%".format((data["Churn"] == "Yes").mean() * 100))
print()
print(
    "Avg tenure — churned: {:.1f} mo | stayed: {:.1f} mo".format(
        data.loc[data["Churn"] == "Yes", "tenure"].mean(),
        data.loc[data["Churn"] == "No", "tenure"].mean(),
    )
)
print(
    "Avg monthly bill — churned: ${:.2f} | stayed: ${:.2f}".format(
        data.loc[data["Churn"] == "Yes", "MonthlyCharges"].mean(),
        data.loc[data["Churn"] == "No", "MonthlyCharges"].mean(),
    )
)
print()
print(
    "Highest churn contract:",
    churn_rate_table("Contract").index[0],
    f"({churn_rate_table('Contract').iloc[0]['Yes']}%)",
)
print(
    "Highest churn internet:",
    churn_rate_table("InternetService").index[0],
    f"({churn_rate_table('InternetService').iloc[0]['Yes']}%)",
)
print(
    "Highest churn payment:",
    churn_rate_table("PaymentMethod").index[0],
    f"({churn_rate_table('PaymentMethod').iloc[0]['Yes']}%)",
)


#### What these takeaway numbers mean / معنی این اعداد جمع‌بندی

**English**

- Dataset size: **7,043** customers and **21** columns.
- Overall churn: about **26.5%**.
- Churners stay much less (**~18 months**) than non-churners (**~38 months**).
- Churners pay higher monthly bills on average (**~$74** vs **~$61**).
- Highest-risk segments: **Month-to-month (~42.7%)**, **Fiber optic (~41.9%)**, **Electronic check (~45.3%)**.

These patterns guide which features matter for the classification model next.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

- اندازه داده: **۷۰۴۳** مشتری و **۲۱** ستون.
- نرخ کل ریزش: حدود **۲۶٫۵٪**.
- ریزش‌کرده‌ها خیلی کمتر می‌مانند (**حدود ۱۸ ماه**) نسبت به مانده‌ها (**حدود ۳۸ ماه**).
- ریزش‌کرده‌ها به‌طور میانگین صورتحساب ماهانه بالاتری دارند (**حدود ۷۴ دلار** در برابر **۶۱ دلار**).
- پرریسک‌ترین بخش‌ها: **قرارداد ماهانه (حدود ۴۲٫۷٪)**، **فیبر نوری (حدود ۴۱٫۹٪)**، **چک الکترونیک (حدود ۴۵٫۳٪)**.

این الگوها نشان می‌دهند کدام ویژگی‌ها برای مدل بعدی مهم‌ترند.

</div>


#### Cleaning: convert TotalCharges / پاک‌سازی: تبدیل TotalCharges

**English**

`TotalCharges` was stored as text. Converting with `errors='coerce'` turns bad/blank values into missing (`NaN`) so we can fix them.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

ستون `TotalCharges` به‌صورت متن ذخیره شده بود. با `errors='coerce'` مقادیر خراب یا خالی به مقدار گمشده (`NaN`) تبدیل می‌شوند تا بعداً درستشان کنیم.

</div>


In [ ]:

# Convert TotalCharges to numeric, coerce invalid values to NaN
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')

In [ ]:
data.info()  # Get information about columns and data types

#### What `data.info()` tells us / `data.info()` چه می‌گوید

**English**

Shows each column’s data type and how many non-null values it has.
After conversion, `TotalCharges` should be numeric and usually show **11** missing values before filling.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

نوع داده هر ستون و تعداد مقادیر غیرخالی را نشان می‌دهد.
بعد از تبدیل، `TotalCharges` باید عددی باشد و معمولاً قبل از پر کردن، **۱۱** مقدار گمشده دارد.

</div>


In [ ]:
data.describe()

#### What `data.describe()` tells us / `data.describe()` چه می‌گوید

**English**

Summary statistics for numeric columns: count, mean, std, min, 25%, 50% (median), 75%, max.
Example: mean tenure **~32.4** months, mean monthly charge **~$64.76**.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

آمار خلاصه برای ستون‌های عددی: تعداد، میانگین، انحراف معیار، کمینه، چارک‌ها، میانه و بیشینه.
مثال: میانگین مدت حدود **۳۲٫۴** ماه و میانگین هزینه ماهانه حدود **۶۴٫۷۶ دلار**.

</div>


In [ ]:
data.isnull().sum()

#### Missing values check / بررسی مقادیر گمشده

**English**

Counts missing values in every column. Expect about **11** missing in `TotalCharges` before imputation; other columns are usually complete.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

تعداد مقادیر گمشده هر ستون را می‌شمارد. قبل از جایگزینی، حدود **۱۱** مقدار گمشده در `TotalCharges` انتظار می‌رود؛ بقیه ستون‌ها معمولاً کامل‌اند.

</div>


In [ ]:
data['TotalCharges'].fillna(data['TotalCharges'].mean(), inplace=True)

#### Fill missing TotalCharges / پر کردن TotalCharges گمشده

**English**

Replaces the **11** missing `TotalCharges` values with the column mean so modeling does not break on `NaN`.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

آن **۱۱** مقدار گمشده `TotalCharges` را با میانگین همان ستون جایگزین می‌کند تا مدل روی `NaN` خطا ندهد.

</div>


In [ ]:
print(data.isnull().sum()) 

#### Confirm no missing values left / تأیید نبودن مقدار گمشده

**English**

Re-check missing counts. After `fillna`, `TotalCharges` should show **0** missing values.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

دوباره تعداد گمشده‌ها را چک می‌کنیم. بعد از `fillna` باید `TotalCharges` صفر مقدار گمشده داشته باشد.

</div>


#### Choose model features / انتخاب ویژگی‌های مدل

**English**

We separate columns into:
- **categorical_cols:** text/category fields (gender, contract, internet, payment method, ...)
- **numerical_cols:** `tenure`, `MonthlyCharges`, `TotalCharges`

The preprocessor will treat these two groups differently.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

ستون‌ها را به دو گروه جدا می‌کنیم:
- **categorical_cols:** فیلدهای متنی/دسته‌ای (جنسیت، قرارداد، اینترنت، روش پرداخت و ...)
- **numerical_cols:** `tenure`، `MonthlyCharges`، `TotalCharges`

پیش‌پردازش برای این دو گروه متفاوت انجام می‌شود.

</div>


In [ ]:
categorical_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 
                    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                    'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']


In [ ]:
data

#### Build, train, and evaluate the model / ساخت، آموزش و ارزیابی مدل

**English**

This is the core modeling cell:
1. **Numeric pipeline:** fill missing with mean + scale with `StandardScaler`.
2. **Categorical pipeline:** fill missing with most frequent + one-hot encode.
3. Combine both in a `ColumnTransformer`.
4. Use **Logistic Regression** as the classifier.
5. Split data **80% train / 20% test** (stratified by Churn).
6. Fit the full pipeline and print accuracy + classification report.

`Churn` and `ChurnFlag` are dropped from `X` so the label is not leaked into features.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

این سلول اصلی مدل‌سازی است:
1. **پایپ‌لاین عددی:** پر کردن گمشده با میانگین + مقیاس‌بندی با `StandardScaler`.
2. **پایپ‌لاین دسته‌ای:** پر کردن گمشده با پرتکرارترین مقدار + کدگذاری one-hot.
3. ترکیب هر دو در `ColumnTransformer`.
4. استفاده از **رگرسیون لجستیک** به‌عنوان مدل.
5. تقسیم داده به **۸۰٪ آموزش / ۲۰٪ آزمون** (با حفظ نسبت ریزش).
6. آموزش کل پایپ‌لاین و چاپ دقت + گزارش طبقه‌بندی.

`Churn` و `ChurnFlag` از `X` حذف می‌شوند تا برچسب هدف به ویژگی‌ها نشت نکند.

</div>


In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', sparse_output=False))
])

# Combine transformers using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# Define the classifier
classifier = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

# Create a pipeline that includes preprocessing and classification
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', classifier)])

# Split the data into training and testing sets
X = data.drop(['Churn', 'ChurnFlag'], axis=1, errors='ignore')
y = data['Churn']
X = X.iloc[:len(y), :]
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)

# Fit the pipeline on the training data
pipeline.fit(X_train, y_train)

# Choose the highest-precision validation threshold that reaches the recall target.
target_recall = 0.80
val_proba = pipeline.predict_proba(X_val)[:, 1]
candidate_thresholds = np.linspace(0.05, 0.95, 181)
threshold_results = []
for threshold in candidate_thresholds:
    val_pred = np.where(val_proba >= threshold, 'Yes', 'No')
    threshold_results.append((
        threshold,
        precision_score(y_val, val_pred, pos_label='Yes', zero_division=0),
        recall_score(y_val, val_pred, pos_label='Yes', zero_division=0),
    ))
eligible = [result for result in threshold_results if result[2] >= target_recall]
decision_threshold, val_precision, val_recall = max(eligible, key=lambda result: result[1])

# Predict and evaluate once on the untouched test set.
y_proba = pipeline.predict_proba(X_test)[:, 1]
y_pred = np.where(y_proba >= decision_threshold, 'Yes', 'No')
accuracy = accuracy_score(y_test, y_pred)
print(f'Validation threshold: {decision_threshold:.2f} (precision={val_precision:.2f}, recall={val_recall:.2f})')
print(f'Accuracy: {accuracy:.2f}')

confusion = confusion_matrix(y_test, y_pred)
print('Confusion Matrix:')
print(confusion)

classification_rep = classification_report(y_test, y_pred)
print('Classification Report:')
print(classification_rep)

model_artifact = {'pipeline': pipeline, 'decision_threshold': decision_threshold}
joblib.dump(model_artifact, MODELS_DIR / 'classifier_model_1.joblib')


## After training: metrics report / بعد از آموزش: گزارش متریک‌ها

**English**

Right after training we measure how good the model is on the **test set**.
We report metrics for **both classes**:
- **No** = customer stayed (no churn)
- **Yes** = customer churned

Key metrics:
- **Accuracy:** share of all predictions that are correct
- **Precision:** of predicted class X, how many were truly X
- **Recall:** of true class X, how many we found
- **F1-score:** balance of precision and recall
- **Support:** how many true samples of that class are in the test set

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

بلافاصله بعد از آموزش، کیفیت مدل را روی **داده آزمون** می‌سنجیم.
متریک‌ها را برای **هر دو کلاس** گزارش می‌کنیم:
- **No** = مشتری مانده (بدون ریزش)
- **Yes** = مشتری ریزش کرده

متریک‌های مهم:
- **Accuracy:** سهم کل پیش‌بینی‌های درست
- **Precision:** از کسانی که مدل گفته کلاس X، چند نفر واقعاً X بوده‌اند
- **Recall:** از کسانی که واقعاً کلاس X بوده‌اند، چند نفر را پیدا کرده‌ایم
- **F1-score:** تعادل بین Precision و Recall
- **Support:** تعداد واقعی آن کلاس در داده آزمون

</div>


In [ ]:
# --- Metrics for BOTH classes (No and Yes) after training ---
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support

y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Accuracy: {accuracy:.3f} ({accuracy*100:.1f}%)")
print("\nClassification report (both classes):")
print(classification_report(y_test, y_pred, digits=3))

# Clean table for No and Yes
precision, recall, f1, support = precision_recall_fscore_support(
    y_test, y_pred, labels=["No", "Yes"], zero_division=0
)
metrics_table = pd.DataFrame(
    {
        "Class": ["No (no churn)", "Yes (churn)"],
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "Support": support,
    }
)
metrics_table[["Precision", "Recall", "F1-score"]] = (
    metrics_table[["Precision", "Recall", "F1-score"]] * 100
).round(1)
metrics_table["Support"] = metrics_table["Support"].astype(int)

# Add macro / weighted averages
macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
    y_test, y_pred, average="macro", zero_division=0
)
weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
    y_test, y_pred, average="weighted", zero_division=0
)

print("\nMetrics table (%):")
print(metrics_table.to_string(index=False))
print(f"\nMacro avg    -> Precision={macro_p*100:.1f}%, Recall={macro_r*100:.1f}%, F1={macro_f1*100:.1f}%")
print(f"Weighted avg -> Precision={weighted_p*100:.1f}%, Recall={weighted_r*100:.1f}%, F1={weighted_f1*100:.1f}%")
print(f"Accuracy     -> {accuracy*100:.1f}%")

metrics_table


#### How to read Accuracy, Precision, Recall, F1 / چطور Accuracy، Precision، Recall و F1 را بخوانیم

**English**

**Overall Accuracy**
- Example idea: if accuracy is ~80%, about 4 of 5 test customers are classified correctly.
- With imbalance (~73.5% No), accuracy alone can look good even if churn detection is weak.

**For class Yes (churn)**
- **Precision (Yes):** when model says churn, how often it is right.
- **Recall (Yes):** of all real churners, how many we caught.
- **F1 (Yes):** harmonic mean of those two.

**For class No (no churn)**
- Same definitions, but for customers who stayed.
- Usually higher than Yes because No is the majority class.

**Macro vs Weighted**
- **Macro:** simple average of No and Yes (treats both classes equally).
- **Weighted:** average weighted by support (closer to overall behavior).

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

**دقت کلی (Accuracy)**
- اگر دقت حدود ۸۰٪ باشد، تقریباً ۴ از ۵ مشتری آزمون درست طبقه‌بندی شده‌اند.
- چون داده نامتوازن است (حدود ۷۳٫۵٪ کلاس No)، Accuracy به‌تنهایی ممکن است خوب به نظر برسد حتی اگر تشخیص ریزش ضعیف باشد.

**برای کلاس Yes (ریزش)**
- **Precision:** وقتی مدل می‌گوید ریزش، چند درصد درست است.
- **Recall:** از همه ریزش‌کرده‌های واقعی، چند درصد را پیدا کرده‌ایم.
- **F1:** میانگین هارمونیک این دو.

**برای کلاس No (مانده)**
- همان تعریف‌ها برای مشتریان مانده.
- معمولاً از Yes بالاتر است چون کلاس اکثریت است.

**Macro و Weighted**
- **Macro:** میانگین ساده No و Yes (هر دو کلاس برابر).
- **Weighted:** میانگین وزن‌دار بر اساس تعداد نمونه (نزدیک‌تر به رفتار کلی).

</div>


In [ ]:
# Bar chart: Precision / Recall / F1 for BOTH classes
plot_df = metrics_table.melt(
    id_vars=["Class"],
    value_vars=["Precision", "Recall", "F1-score"],
    var_name="Metric",
    value_name="Score (%)",
)

plt.figure(figsize=(9, 5))
sns.barplot(data=plot_df, x="Metric", y="Score (%)", hue="Class")
plt.ylim(0, 100)
plt.title("Precision, Recall, F1 for No-churn and Churn")
plt.legend(title="Class")
plt.tight_layout()
plt.show()


#### Plot — metrics for both classes / نمودار — متریک‌ها برای هر دو کلاس

**English**

Each group of bars is one metric.
- Blue/orange bars compare **No (no churn)** vs **Yes (churn)**.
- If Yes bars are much lower than No bars, the model is weaker at detecting churners than stayers.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

هر گروه میله یک متریک است.
- رنگ‌ها کلاس **No (بدون ریزش)** و **Yes (ریزش)** را مقایسه می‌کنند.
- اگر میله‌های Yes خیلی پایین‌تر از No باشند، مدل در پیدا کردن ریزش‌کرده‌ها ضعیف‌تر از تشخیص مانده‌هاست.

</div>


## Confusion matrix / ماتریس درهم‌ریختگی

**English**

Next we look at the confusion matrix: exact counts of correct and incorrect predictions for both classes.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

حالا ماتریس درهم‌ریختگی را می‌بینیم: تعداد دقیق پیش‌بینی‌های درست و غلط برای هر دو کلاس.

</div>


In [ ]:
# Confusion matrix for both classes (counts + percentages)
from sklearn.metrics import confusion_matrix

labels = ["No", "Yes"]
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(
    cm,
    index=["Actual No", "Actual Yes"],
    columns=["Predicted No", "Predicted Yes"],
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", ax=axes[0])
axes[0].set_title("Confusion matrix (counts)")

cm_row_pct = cm / cm.sum(axis=1, keepdims=True)
cm_row_df = pd.DataFrame(
    cm_row_pct,
    index=["Actual No", "Actual Yes"],
    columns=["Predicted No", "Predicted Yes"],
)
sns.heatmap(cm_row_df, annot=True, fmt=".1%", cmap="Oranges", ax=axes[1])
axes[1].set_title("Confusion matrix (recall view: % within each actual class)")

plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print("Breakdown:")
print(f"TN (Actual No, Pred No)  = {tn}")
print(f"FP (Actual No, Pred Yes) = {fp}  <- false alarms")
print(f"FN (Actual Yes, Pred No) = {fn}  <- missed churners")
print(f"TP (Actual Yes, Pred Yes)= {tp}")
print(f"\nRecall No  (TN/(TN+FP)) = {tn/(tn+fp):.3f}")
print(f"Recall Yes (TP/(TP+FN)) = {tp/(tp+fn):.3f}")
print(f"Precision No  (TN/(TN+FN)) = {tn/(tn+fn):.3f}")
print(f"Precision Yes (TP/(TP+FP)) = {tp/(tp+fp):.3f}")

cm_df


In [ ]:
114+167

#### How to read the confusion matrix / چطور ماتریس درهم‌ریختگی را بخوانیم

**English**

- **TN:** stayed & predicted stay
- **TP:** churned & predicted churn
- **FP:** stayed but predicted churn (false alarm)
- **FN:** churned but predicted stay (missed churner)
- Left plot = raw counts
- Right plot = % within each actual class (row-normalized) → directly related to **Recall** for No and Yes

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

- **TN:** مانده و مدل هم گفته مانده
- **TP:** ریزش کرده و مدل هم گفته ریزش
- **FP:** مانده ولی مدل گفته ریزش (هشدار اشتباه)
- **FN:** ریزش کرده ولی مدل گفته مانده (ریزش ازدست‌رفته)
- نمودار چپ = تعداد خام
- نمودار راست = درصد داخل هر کلاس واقعی (نرمال سطری) → مستقیماً به **Recall** کلاس No و Yes مربوط است

</div>


## Complete evaluation plots for both classes / نمودارهای کامل ارزیابی برای هر دو کلاس

**English**

Below we plot evaluation curves/metrics covering **both churn and no-churn**:
ROC-style one-vs-rest scores, Precision-Recall for each class, and a side-by-side metric dashboard.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

در ادامه منحنی‌ها و متریک‌های ارزیابی را برای **هر دو کلاس ریزش و بدون ریزش** رسم می‌کنیم:
امتیازهای ROC یک‌دربرابر-بقیه، Precision-Recall برای هر کلاس، و داشبورد متریک‌ها در کنار هم.

</div>


In [ ]:
# Complete evaluation plots for BOTH classes (No and Yes)
from sklearn.metrics import (
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
    roc_auc_score,
)

# Binary indicators and probabilities for each class
y_true_no = (y_test == "No").astype(int)
y_true_yes = (y_test == "Yes").astype(int)
proba_yes = y_proba
proba_no = 1.0 - y_proba

# ROC for each class
fpr_yes, tpr_yes, _ = roc_curve(y_true_yes, proba_yes)
fpr_no, tpr_no, _ = roc_curve(y_true_no, proba_no)
auc_yes = auc(fpr_yes, tpr_yes)
auc_no = auc(fpr_no, tpr_no)

# Precision-Recall for each class
prec_yes, rec_yes, _ = precision_recall_curve(y_true_yes, proba_yes)
prec_no, rec_no, _ = precision_recall_curve(y_true_no, proba_no)
ap_yes = average_precision_score(y_true_yes, proba_yes)
ap_no = average_precision_score(y_true_no, proba_no)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1) ROC both classes
axes[0, 0].plot(fpr_yes, tpr_yes, label=f"Yes (AUC={auc_yes:.3f})", color="salmon")
axes[0, 0].plot(fpr_no, tpr_no, label=f"No (AUC={auc_no:.3f})", color="steelblue")
axes[0, 0].plot([0, 1], [0, 1], "k--", label="Random")
axes[0, 0].set_xlabel("False Positive Rate")
axes[0, 0].set_ylabel("True Positive Rate")
axes[0, 0].set_title("ROC curves for both classes")
axes[0, 0].legend(loc="lower right")
axes[0, 0].grid(True, alpha=0.3)

# 2) Precision-Recall both classes
axes[0, 1].plot(rec_yes, prec_yes, label=f"Yes (AP={ap_yes:.3f})", color="salmon")
axes[0, 1].plot(rec_no, prec_no, label=f"No (AP={ap_no:.3f})", color="steelblue")
axes[0, 1].set_xlabel("Recall")
axes[0, 1].set_ylabel("Precision")
axes[0, 1].set_title("Precision-Recall curves for both classes")
axes[0, 1].legend(loc="lower left")
axes[0, 1].grid(True, alpha=0.3)

# 3) Metric dashboard bars already computed
x = np.arange(3)
width = 0.35
axes[1, 0].bar(x - width/2, metrics_table.loc[0, ["Precision", "Recall", "F1-score"]], width, label="No", color="steelblue")
axes[1, 0].bar(x + width/2, metrics_table.loc[1, ["Precision", "Recall", "F1-score"]], width, label="Yes", color="salmon")
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(["Precision", "Recall", "F1"])
axes[1, 0].set_ylim(0, 100)
axes[1, 0].set_ylabel("Score (%)")
axes[1, 0].set_title("Precision / Recall / F1 by class")
axes[1, 0].legend()
axes[1, 0].grid(True, axis="y", alpha=0.3)

# 4) Support + accuracy summary pie-ish bars
support_vals = metrics_table["Support"].values
axes[1, 1].bar(["No support", "Yes support"], support_vals, color=["steelblue", "salmon"])
axes[1, 1].set_title(f"Test support by class | Accuracy={accuracy*100:.1f}%")
axes[1, 1].set_ylabel("Number of customers")
for i, v in enumerate(support_vals):
    axes[1, 1].text(i, v + max(support_vals)*0.01, str(v), ha="center")
axes[1, 1].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("ROC AUC Yes:", round(auc_yes, 3), "| ROC AUC No:", round(auc_no, 3))
print("Average Precision Yes:", round(ap_yes, 3), "| Average Precision No:", round(ap_no, 3))
print("Overall ROC AUC (Yes as positive):", round(roc_auc_score(y_true_yes, proba_yes), 3))


#### How to read the complete evaluation dashboard / چطور داشبورد کامل ارزیابی را بخوانیم

**English**

**Top-left — ROC (both classes)**
- Higher curve / higher AUC = better ranking for that class.
- AUC close to 1 is strong; 0.5 is random.

**Top-right — Precision-Recall (both classes)**
- Especially important for **Yes** because churn is the minority class.
- AP (Average Precision) summarizes the whole PR curve.

**Bottom-left — Precision / Recall / F1 bars**
- Direct comparison of No vs Yes classification quality.

**Bottom-right — Support**
- How many test customers are in each true class (imbalance reminder).

Use this block right after training to judge whether the model is good for **both** stay and churn decisions.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

**بالا-چپ — ROC (هر دو کلاس)**
- منحنی بالاتر / AUC بالاتر = رتبه‌بندی بهتر برای آن کلاس.
- AUC نزدیک ۱ قوی است؛ ۰٫۵ مثل حدس تصادفی است.

**بالا-راست — Precision-Recall (هر دو کلاس)**
- برای کلاس **Yes** خیلی مهم است چون ریزش کلاس اقلیت است.
- AP کل منحنی PR را خلاصه می‌کند.

**پایین-چپ — میله‌های Precision / Recall / F1**
- مقایسه مستقیم کیفیت طبقه‌بندی No در برابر Yes.

**پایین-راست — Support**
- تعداد مشتریان آزمون در هر کلاس واقعی (یادآوری نامتوازنی).

از این بلوک بلافاصله بعد از آموزش استفاده کنید تا ببینید مدل برای **هر دو** تصمیم ماندن و ریزش خوب عمل می‌کند یا نه.

</div>


In [ ]:
X_test_subset = X_test.iloc[0:10]

# Make predictions on the subset of test data using the trained model
y_pred_subset = pipeline.predict(X_test_subset)

# Print the predicted values for the subset
print("Predicted values for the subset:")
print(y_pred_subset)


#### Quick predictions on 10 test rows / پیش‌بینی سریع روی ۱۰ ردیف آزمون

**English**

Takes the first **10** test customers and prints the model’s Yes/No predictions.
Useful for a fast sanity check of raw outputs.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

۱۰ مشتری اول داده آزمون را می‌گیرد و پیش‌بینی بله/خیر مدل را چاپ می‌کند.
برای بررسی سریع خروجی خام مدل مفید است.

</div>


In [ ]:
X_train_subset = X_train.iloc[0:10]

# Make predictions on the subset of test data using the trained model
y_pred_subset = pipeline.predict(X_test_subset)

# Print the predicted values for the subset
print("Predicted values for the subset:")
print(y_pred_subset)


#### Quick check on training rows / بررسی سریع روی ردیف‌های آموزش

**English**

Similar quick prediction check. (This cell currently still predicts on `X_test_subset`; conceptually it is for inspecting a small batch of predictions.)

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

بررسی سریع مشابه. (در کد فعلی هنوز روی `X_test_subset` پیش‌بینی می‌کند؛ از نظر مفهومی برای دیدن یک دسته کوچک از خروجی‌هاست.)

</div>


In [ ]:
pipeline.predict_proba(X_test)

#### Prediction probabilities / احتمال‌های پیش‌بینی

**English**

`predict_proba` returns two columns for each customer: **P(No)** and **P(Yes)**.
A higher **P(Yes)** means the model is more confident the customer will churn.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

`predict_proba` برای هر مشتری دو ستون برمی‌گرداند: **احتمال خیر** و **احتمال بله**.
هرچه **احتمال بله** بالاتر باشد، مدل بیشتر مطمئن است که مشتری ریزش می‌کند.

</div>


In [ ]:
# let's check only the second column (positive probability of churn)
pipeline.predict_proba(X_test)[:, 1]

#### Keep only churn probability / فقط احتمال ریزش را نگه داریم

**English**

`[:, 1]` keeps the second column only = probability of class **Yes** (churn).
This vector is useful for ROC curves and choosing a decision threshold.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

`[:, 1]` فقط ستون دوم را نگه می‌دارد = احتمال کلاس **بله** (ریزش).
این بردار برای منحنی ROC و انتخاب آستانه تصمیم مفید است.

</div>


In [ ]:
# y_pred = pipeline.predict_proba(X_test)[:, 1]
# y_pred >= 0.5
y_pred = pipeline.predict(X_test)
y_pred

comparison_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})


# Print the comparison
# print(comparison_df.head(20))  # Print the first 10 rows for a quick v

#### Create Actual vs Predicted table / ساخت جدول واقعیت در برابر پیش‌بینی

**English**

Builds a DataFrame with true labels (`Actual`) and model labels (`Predicted`) for the test set.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

یک جدول با برچسب واقعی (`Actual`) و برچسب مدل (`Predicted`) برای داده آزمون می‌سازد.

</div>


In [ ]:
comparison_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred,'correct':y_test==y_pred})

#### Add a correctness flag / افزودن ستون درست/نادرست

**English**

Adds `correct = True/False` for each test row by comparing Actual vs Predicted.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

ستون `correct` را اضافه می‌کند: برای هر ردیف آزمون با مقایسه واقعیت و پیش‌بینی، درست یا نادرست بودن مشخص می‌شود.

</div>


In [ ]:


comparison_df.head(20)

#### Inspect the first 20 comparisons / مشاهده ۲۰ مقایسه اول

**English**

Shows the first 20 rows so you can visually spot correct predictions and mistakes.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

۲۰ ردیف اول را نشان می‌دهد تا بتوانید پیش‌بینی‌های درست و اشتباه را سریع ببینید.

</div>


In [ ]:
# Show same as accuracy
comparison_df['correct'].mean()

#### Mean of `correct` = accuracy / میانگین ستون `correct` همان دقت است

**English**

The average of True/False values equals the share of correct predictions — the same idea as accuracy.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

میانگین مقادیر True/False برابر است با سهم پیش‌بینی‌های درست؛ یعنی همان مفهوم دقت (Accuracy).

</div>


#### Full model evaluation and plots / ارزیابی کامل مدل و نمودارها

**English**

This cell computes accuracy, precision/recall/F1 for the **Yes** class, confusion matrix, classification report, then plots:
1. ROC curve
2. Precision-Recall curve
3. Confusion-matrix heatmap

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

این سلول دقت، precision/recall/F1 برای کلاس **بله**، ماتریس درهم‌ریختگی و گزارش طبقه‌بندی را حساب می‌کند و سپس این نمودارها را می‌کشد:
1. منحنی ROC
2. منحنی Precision-Recall
3. نقشه ماتریس درهم‌ریختگی

</div>


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
confusion = confusion_matrix(y_test, y_pred)
classification_rep = classification_report(y_test, y_pred)

    # Calculate precision for the 'Yes' class
precision_yes = precision_score(y_test, y_pred, pos_label='Yes')

    # Calculate recall for the 'Yes' class
recall_yes = recall_score(y_test, y_pred, pos_label='Yes')

    # Calculate F1-score for the 'Yes' class
f1_yes = f1_score(y_test, y_pred, pos_label='Yes')

    # Calculate the confusion matrix
confusion = confusion_matrix(y_test, y_pred, labels=['No', 'Yes'])

    # Print the evaluation metrics for the 'Yes' class
print(f'classifier: {classifier}')
print(f'Accuracy: {accuracy:.2f}')
print(f'Precision for "Yes" class: {precision_yes:.2f}')
print(f'Recall for "Yes" class: {recall_yes:.2f}')
print(f'F1-score for "Yes" class: {f1_yes:.2f}')
print('Confusion Matrix:')
print(confusion)
print(f'report: {classification_rep}')



    #  Visual representation of your model's performance (e.g., ROC curve, confusion matrix).
label_mapping = {'No': 0, 'Yes': 1}
y_test_binary = [label_mapping[label] for label in y_test]
y_pred_binary = [label_mapping[label] for label in y_pred]
    # Compute ROC curve and ROC area
fpr, tpr, _ = roc_curve(y_test_binary, y_pred_binary)
roc_auc = auc(fpr, tpr)

    # Compute Precision-Recall curve and area
precision, recall, _ = precision_recall_curve(y_test_binary, y_pred_binary)
pr_auc = average_precision_score(y_test_binary, y_pred_binary)

    # Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.show()

    # Plot Precision-Recall curve
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='darkorange', lw=2, label=f'Precision-Recall curve (AUC = {pr_auc:.2f})')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='lower left')
plt.show()

plt.figure(figsize=(8, 6))
sns.heatmap(confusion, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.title("Confusion Matrix")
plt.show()

#### How to read the evaluation plots / چطور نمودارهای ارزیابی را بخوانیم

**English**

- **ROC:** above the diagonal is better; AUC near **1** is strong, **0.5** is random.
- **Precision-Recall:** especially useful because churn is only **~26.5%** of customers.
- **Confusion matrix:** diagonal = correct; off-diagonal = mistakes (false alarms / missed churners).
- Prefer **Yes** precision/recall over accuracy alone for this imbalanced problem.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

- **ROC:** بالاتر از خط قطری بهتر است؛ AUC نزدیک **۱** قوی و **۰٫۵** مثل حدس تصادفی است.
- **Precision-Recall:** چون فقط حدود **۲۶٫۵٪** مشتری ریزش می‌کنند خیلی مفید است.
- **ماتریس درهم‌ریختگی:** قطر اصلی = درست؛ خارج قطر = اشتباه (هشدار غلط / از دست دادن ریزش‌کرده).
- در این مسئله نامتوازن، precision/recall کلاس **بله** از Accuracy تنها مهم‌تر است.

</div>


#### Learning curve / منحنی یادگیری

**English**

Shows how train and validation accuracy change as the model sees more training examples (5-fold CV).

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

نشان می‌دهد با بیشتر شدن داده آموزش، دقت آموزش و اعتبارسنجی چطور تغییر می‌کند (اعتبارسنجی ۵ بخشی).

</div>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import learning_curve

# Define a function to plot the learning curve for accuracy
def plot_learning_curve_accuracy(estimator, X, y, train_sizes=np.linspace(0.1, 1.0, 10), cv=None):
    train_sizes, train_scores, test_scores = learning_curve(
        estimator, X, y, cv=cv, scoring='accuracy', train_sizes=train_sizes)

    train_scores_mean = np.mean(train_scores, axis=1)
    test_scores_mean = np.mean(test_scores, axis=1)

    plt.figure(figsize=(8, 6))
    plt.title("Learning Curve for Accuracy")
    plt.xlabel("Number of Training Examples")
    plt.ylabel("Accuracy")
    plt.grid()

    plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Training Accuracy")
    plt.plot(train_sizes, test_scores_mean, 'o-', color="g", label="Test Accuracy")

    plt.legend(loc="best")
    plt.show()

# Assuming you have your dataset X and y
# estimator is your trained classification model (e.g., logistic regression, random forest)
# cv is the number of cross-validation folds (e.g., 5 or 10)

# Plot the learning curve for accuracy
plot_learning_curve_accuracy(pipeline, X_train, y_train, cv=5)

#### How to read the learning curve / چطور منحنی یادگیری را بخوانیم

**English**

- **Red (training):** how well the model fits training data.
- **Green (test/CV):** how well it generalizes.
- Both high and close → good learning, little overfitting.
- Red high, green much lower → overfitting.
- Both low → underfitting.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

- **قرمز (آموزش):** مدل چقدر روی داده آموزش خوب برازش می‌شود.
- **سبز (آزمون/اعتبارسنجی):** چقدر خوب تعمیم می‌دهد.
- هر دو بالا و نزدیک هم → یادگیری خوب و بیش‌برازش کم.
- قرمز بالا و سبز خیلی پایین‌تر → بیش‌برازش.
- هر دو پایین → کم‌برازش.

</div>


## More prediction plots and useful insights / نمودارهای بیشتر پیش‌بینی و بینش‌های مفید

**English**

These cells go beyond basic accuracy. They help you:
- see how confident the model is
- choose a better decision threshold
- find which features push churn up or down
- list the highest-risk customers for action

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

این سلول‌ها فراتر از دقت ساده هستند و کمک می‌کنند:
- ببینید مدل چقدر مطمئن است
- آستانه تصمیم بهتری انتخاب کنید
- بفهمید کدام ویژگی‌ها ریزش را بالا یا پایین می‌برند
- پرریسک‌ترین مشتریان را برای اقدام پیدا کنید

</div>


In [ ]:
# Probabilities for the positive class (Churn = Yes)
y_proba = pipeline.predict_proba(X_test)[:, 1]

pred_df = X_test.copy()
pred_df["Actual"] = y_test.values
pred_df["Predicted"] = y_pred
pred_df["ChurnProb"] = y_proba
pred_df["Correct"] = pred_df["Actual"] == pred_df["Predicted"]

print(pred_df[["Actual", "Predicted", "ChurnProb", "Correct"]].head(10))
print("\nChurnProb summary:")
print(pred_df["ChurnProb"].describe().round(3))


#### Prediction table with churn probability / جدول پیش‌بینی همراه با احتمال ریزش

**English**

For each test customer we keep:
- **Actual** true label
- **Predicted** model label at default threshold 0.5
- **ChurnProb** probability of Yes
- **Correct** whether prediction matched reality

Higher `ChurnProb` = model thinks churn is more likely.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

برای هر مشتری آزمون نگه می‌داریم:
- **Actual** برچسب واقعی
- **Predicted** برچسب مدل با آستانه پیش‌فرض ۰٫۵
- **ChurnProb** احتمال کلاس بله
- **Correct** آیا پیش‌بینی درست بوده است

`ChurnProb` بالاتر یعنی مدل ریزش را محتمل‌تر می‌داند.

</div>


In [ ]:
# Plot 1: distribution of predicted churn probabilities by actual class
plt.figure(figsize=(8, 4))
sns.histplot(
    data=pred_df,
    x="ChurnProb",
    hue="Actual",
    bins=25,
    element="step",
    stat="density",
    common_norm=False,
)
plt.axvline(0.5, color="black", linestyle="--", label="Default threshold = 0.5")
plt.title("Predicted churn probability by actual class")
plt.xlabel("Predicted probability of Churn=Yes")
plt.ylabel("Density")
plt.legend(title="Actual")
plt.tight_layout()
plt.show()


#### Plot — probability distribution / نمودار — توزیع احتمال پیش‌بینی

**English**

- Curves/histograms show where Actual=Yes and Actual=No sit on the probability axis.
- A good model pushes real churners toward the right (high probability) and non-churners toward the left.
- The dashed line is the default cut-off **0.5** used by `predict()`.
- If many real churners are left of 0.5, lowering the threshold can catch more churners (higher recall).

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

- این نمودار نشان می‌دهد مشتریان واقعی Yes و No روی محور احتمال کجا قرار گرفته‌اند.
- مدل خوب، ریزش‌کرده‌های واقعی را به سمت راست (احتمال بالا) و مانده‌ها را به سمت چپ می‌برد.
- خط‌چین، آستانه پیش‌فرض **۰٫۵** در `predict()` است.
- اگر خیلی از ریزش‌کرده‌های واقعی چپِ ۰٫۵ باشند، کم کردن آستانه می‌تواند ریزش‌های بیشتری را پیدا کند (Recall بالاتر).

</div>


In [ ]:
# Plot 2: clearer confusion matrix with labels and percentages
cm = confusion_matrix(y_test, y_pred, labels=["No", "Yes"])
cm_df = pd.DataFrame(cm, index=["Actual No", "Actual Yes"], columns=["Pred No", "Pred Yes"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", ax=axes[0])
axes[0].set_title("Confusion matrix (counts)")

cm_pct = cm / cm.sum()
sns.heatmap(
    pd.DataFrame(cm_pct, index=["Actual No", "Actual Yes"], columns=["Pred No", "Pred Yes"]),
    annot=True,
    fmt=".1%",
    cmap="Oranges",
    ax=axes[1],
)
axes[1].set_title("Confusion matrix (share of test set)")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives (stayed, predicted stay): {tn}")
print(f"False Positives (stayed, predicted churn): {fp}")
print(f"False Negatives (churned, predicted stay): {fn}")
print(f"True Positives (churned, predicted churn): {tp}")
print(f"False Negative rate among actual churners: {fn / (fn + tp):.1%}")


#### Plot — confusion matrix counts and shares / نمودار — ماتریس درهم‌ریختگی تعداد و درصد

**English**

- **True Negatives:** correctly predicted No
- **True Positives:** correctly predicted Yes
- **False Positives:** false alarms (said Yes, actually No)
- **False Negatives:** missed churners (said No, actually Yes) — often costly for business
- Right heatmap shows each cell as % of the whole test set

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

- **True Negatives:** درست پیش‌بینی شده که نمی‌رود
- **True Positives:** درست پیش‌بینی شده که می‌رود
- **False Positives:** هشدار اشتباه (گفته می‌رود، در واقع مانده)
- **False Negatives:** ریزش ازدست‌رفته (گفته می‌ماند، در واقع رفته) — معمولاً برای کسب‌وکار هزینه‌بر است
- نقشه سمت راست، سهم هر خانه از کل داده آزمون را نشان می‌دهد

</div>


In [ ]:
# Plot 3: threshold analysis — precision / recall / F1 vs decision threshold
thresholds = np.linspace(0.1, 0.9, 33)
precisions, recalls, f1s, accuracies = [], [], [], []

for t in thresholds:
    y_hat = np.where(y_proba >= t, "Yes", "No")
    precisions.append(precision_score(y_test, y_hat, pos_label="Yes", zero_division=0))
    recalls.append(recall_score(y_test, y_hat, pos_label="Yes", zero_division=0))
    f1s.append(f1_score(y_test, y_hat, pos_label="Yes", zero_division=0))
    accuracies.append(accuracy_score(y_test, y_hat))

plt.figure(figsize=(9, 5))
plt.plot(thresholds, precisions, label="Precision (Yes)")
plt.plot(thresholds, recalls, label="Recall (Yes)")
plt.plot(thresholds, f1s, label="F1 (Yes)")
plt.plot(thresholds, accuracies, label="Accuracy", linestyle="--")
plt.axvline(0.5, color="black", linestyle=":", label="Default 0.5")
best_f1_idx = int(np.argmax(f1s))
plt.axvline(thresholds[best_f1_idx], color="green", linestyle="--", label=f"Best F1 @ {thresholds[best_f1_idx]:.2f}")
plt.xlabel("Decision threshold on ChurnProb")
plt.ylabel("Score")
plt.title("Threshold tuning for churn detection")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Best F1 threshold: {thresholds[best_f1_idx]:.2f}")
print(f"At that threshold -> Precision={precisions[best_f1_idx]:.3f}, Recall={recalls[best_f1_idx]:.3f}, F1={f1s[best_f1_idx]:.3f}")


#### Plot — threshold tuning / نمودار — تنظیم آستانه تصمیم

**English**

Default `predict()` uses **0.5**. This plot shows what happens if you change it.
- Lower threshold → catch more churners (**higher recall**), but more false alarms (**lower precision**)
- Higher threshold → fewer false alarms (**higher precision**), but miss more churners (**lower recall**)
- Green line marks the threshold with the best **F1** for class Yes on the test set

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

تابع `predict()` به‌صورت پیش‌فرض از آستانه **۰٫۵** استفاده می‌کند. این نمودار اثر تغییر آن را نشان می‌دهد.
- آستانه پایین‌تر → ریزش‌های بیشتری پیدا می‌شود (**Recall بالاتر**) ولی هشدار اشتباه بیشتر می‌شود (**Precision پایین‌تر**)
- آستانه بالاتر → هشدار اشتباه کمتر (**Precision بالاتر**) ولی ریزش‌های بیشتری از دست می‌رود (**Recall پایین‌تر**)
- خط سبز، آستانه‌ای است که بهترین **F1** کلاس بله را روی داده آزمون می‌دهد

</div>


In [ ]:
# Plot 4: feature importance from Logistic Regression coefficients
ohe = pipeline.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]
cat_feature_names = ohe.get_feature_names_out(categorical_cols)
feature_names = np.concatenate([numerical_cols, cat_feature_names])

coefs = pipeline.named_steps["classifier"].coef_[0]
coef_df = pd.DataFrame({"feature": feature_names, "coefficient": coefs})
coef_df["abs_coef"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values("abs_coef", ascending=False)

top_n = 15
top_coef = coef_df.head(top_n).sort_values("coefficient")

plt.figure(figsize=(9, 6))
colors = ["salmon" if v > 0 else "steelblue" for v in top_coef["coefficient"]]
plt.barh(top_coef["feature"], top_coef["coefficient"], color=colors)
plt.axvline(0, color="black", linewidth=1)
plt.title(f"Top {top_n} Logistic Regression coefficients")
plt.xlabel("Coefficient (positive = more likely Churn=Yes)")
plt.tight_layout()
plt.show()

print("Top features increasing churn risk (positive coef):")
print(coef_df[coef_df["coefficient"] > 0].head(8)[["feature", "coefficient"]].to_string(index=False))
print("\nTop features decreasing churn risk (negative coef):")
print(coef_df[coef_df["coefficient"] < 0].head(8)[["feature", "coefficient"]].to_string(index=False))


#### Plot — feature importance (coefficients) / نمودار — اهمیت ویژگی‌ها (ضرایب مدل)

**English**

Logistic Regression coefficients show direction of effect (after preprocessing):
- **Positive (salmon):** feature associated with **higher** churn probability
- **Negative (steelblue):** feature associated with **lower** churn probability
- Larger absolute value → stronger effect in this linear model

Useful for explaining *why* the model flags someone as high risk.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

ضرایب رگرسیون لجستیک جهت اثر را نشان می‌دهند (بعد از پیش‌پردازش):
- **مثبت (صورتی/مرجانی):** ویژگی با **احتمال بالاتر** ریزش مرتبط است
- **منفی (آبی):** ویژگی با **احتمال پایین‌تر** ریزش مرتبط است
- قدرمطلق بزرگ‌تر → اثر قوی‌تر در این مدل خطی

برای توضیح اینکه *چرا* مدل کسی را پرریسک می‌داند مفید است.

</div>


In [ ]:
# Plot 5: risk buckets — actual churn rate by predicted probability bin
pred_df["RiskBin"] = pd.cut(
    pred_df["ChurnProb"],
    bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0],
    labels=["0-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1.0"],
    include_lowest=True,
)

risk_table = (
    pred_df.groupby("RiskBin", observed=False)
    .agg(
        customers=("ChurnProb", "size"),
        avg_pred_prob=("ChurnProb", "mean"),
        actual_churn_rate=("Actual", lambda s: (s == "Yes").mean()),
    )
    .round(3)
)
risk_table["actual_churn_rate_pct"] = (risk_table["actual_churn_rate"] * 100).round(1)
print(risk_table)

plt.figure(figsize=(8, 4))
plt.bar(risk_table.index.astype(str), risk_table["actual_churn_rate_pct"], color="salmon")
plt.title("Actual churn rate (%) by predicted risk bin")
plt.xlabel("Predicted probability bin")
plt.ylabel("Actual churn rate (%)")
plt.tight_layout()
plt.show()


#### Plot — risk bins (model lift check) / نمودار — سبدهای ریسک (بررسی کیفیت رتبه‌بندی)

**English**

Customers are grouped by predicted probability.
If the model ranks well, actual churn % should rise from left (low risk) to right (high risk).
This is very useful for campaigns: focus retention offers on the highest bins first.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

مشتریان بر اساس احتمال پیش‌بینی‌شده گروه‌بندی می‌شوند.
اگر مدل خوب رتبه‌بندی کند، درصد ریزش واقعی از چپ (کم‌ریسک) به راست (پرریسک) بالا می‌رود.
برای کمپین نگهداشت خیلی مفید است: اول روی سبدهای پرریسک تمرکز کنید.

</div>


In [ ]:
# Useful list: highest-risk customers in the test set
top_risk = (
    pred_df.sort_values("ChurnProb", ascending=False)
    [["customerID", "Contract", "InternetService", "tenure", "MonthlyCharges", "PaymentMethod", "Actual", "Predicted", "ChurnProb"]]
    .head(15)
)
print("Top 15 highest predicted churn probabilities:")
display_cols = top_risk.copy()
display_cols["ChurnProb"] = display_cols["ChurnProb"].round(3)
print(display_cols.to_string(index=False))

# Error breakdown
fp_n = int(((pred_df["Actual"] == "No") & (pred_df["Predicted"] == "Yes")).sum())
fn_n = int(((pred_df["Actual"] == "Yes") & (pred_df["Predicted"] == "No")).sum())
print(f"\nFalse alarms (FP): {fp_n}")
print(f"Missed churners (FN): {fn_n}")
print(f"Mean ChurnProb for actual Yes: {pred_df.loc[pred_df['Actual']=='Yes','ChurnProb'].mean():.3f}")
print(f"Mean ChurnProb for actual No : {pred_df.loc[pred_df['Actual']=='No','ChurnProb'].mean():.3f}")


#### High-risk customer list and error summary / لیست مشتریان پرریسک و خلاصه خطاها

**English**

- The table ranks test customers by `ChurnProb` (highest first).
- Use it as an action list for retention calls/offers.
- Compare `Actual` vs `Predicted` to see which high-risk cases the model already gets right.
- FP = wasted outreach; FN = missed opportunity to save a customer.
- Mean probability should be higher for Actual=Yes than Actual=No if the model separates classes well.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

- جدول مشتریان آزمون را بر اساس `ChurnProb` از زیاد به کم رتبه‌بندی می‌کند.
- می‌توانید آن را به‌عنوان لیست اقدام برای تماس/پیشنهاد نگهداشت استفاده کنید.
- با مقایسه `Actual` و `Predicted` ببینید مدل در موارد پرریسک کجا درست کار می‌کند.
- FP = تماس هدررفته؛ FN = فرصت ازدست‌رفته برای حفظ مشتری.
- اگر مدل خوب جدا کند، میانگین احتمال برای Actual=Yes باید از Actual=No بالاتر باشد.

</div>


In [ ]:
# Final useful prediction summary
from sklearn.metrics import roc_auc_score

roc_auc_val = roc_auc_score((y_test == "Yes").astype(int), y_proba)
ap_val = average_precision_score((y_test == "Yes").astype(int), y_proba)

print("===== PREDICTION SUMMARY =====")
print(f"Test size: {len(y_test)}")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision (Yes): {precision_score(y_test, y_pred, pos_label='Yes'):.3f}")
print(f"Recall (Yes): {recall_score(y_test, y_pred, pos_label='Yes'):.3f}")
print(f"F1 (Yes): {f1_score(y_test, y_pred, pos_label='Yes'):.3f}")
print(f"ROC AUC: {roc_auc_val:.3f}")
print(f"Average Precision: {ap_val:.3f}")
print(f"Best F1 threshold: {thresholds[best_f1_idx]:.2f}")
print("==============================")


#### Final prediction summary / جمع‌بندی نهایی پیش‌بینی

**English**

This block prints the most useful model scores in one place:
- **Accuracy:** overall correctness
- **Precision/Recall/F1 (Yes):** quality on the churn class
- **ROC AUC:** ranking quality across thresholds
- **Average Precision:** ranking quality with class imbalance in mind
- **Best F1 threshold:** a practical alternative to 0.5 for catching churners

Use high-risk bins + top-risk list for business action; use threshold/F1 for model operating point.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

این بلوک مهم‌ترین امتیازهای مدل را یکجا چاپ می‌کند:
- **Accuracy:** صحت کلی
- **Precision/Recall/F1 (Yes):** کیفیت روی کلاس ریزش
- **ROC AUC:** کیفیت رتبه‌بندی در آستانه‌های مختلف
- **Average Precision:** کیفیت رتبه‌بندی با در نظر گرفتن نامتوازن بودن کلاس‌ها
- **Best F1 threshold:** جایگزین کاربردی برای ۰٫۵ جهت پیدا کردن ریزش‌کرده‌ها

برای اقدام کسب‌وکار از سبدهای پرریسک و لیست top-risk استفاده کنید؛ برای نقطه کار مدل از آستانه/F1.

</div>


## Random Forest and XGBoost / جنگل تصادفی و XGBoost

**English**

The cells below train **Random Forest** and **XGBoost** on the **same** 80/20 stratified split used for Logistic Regression (`X_train`, `X_test`, `y_train`, `y_test`).  
The same `preprocessor` is reused (mean-impute + scale numerics, most-frequent + one-hot categoricals). Then we evaluate both models and show predictions (labels + churn probabilities).

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

سلول‌های زیر **Random Forest** و **XGBoost** را روی **همان** تقسیم ۸۰/۲۰ آموزش می‌دهند که برای رگرسیون لجستیک استفاده شد.  
همان پیش‌پردازش اعمال می‌شود. سپس هر دو مدل ارزیابی می‌شوند و پیش‌بینی‌ها (برچسب + احتمال ریزش) نشان داده می‌شود.

</div>


In [ ]:
# Train Random Forest and XGBoost on the same split / preprocessor as Logistic Regression
n_no = int((y_train == "No").sum())
n_yes = int((y_train == "Yes").sum())
# XGBoost 3.x needs numeric labels (0/1), not "No"/"Yes"
y_train_bin = (y_train == "Yes").astype(int)

rf_pipeline = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", RandomForestClassifier(
        n_estimators=400,
        max_depth=12,
        min_samples_leaf=5,
        min_samples_split=10,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )),
])

xgb_pipeline = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        objective="binary:logistic",
        eval_metric="auc",
        scale_pos_weight=n_no / max(n_yes, 1),
        random_state=42,
        n_jobs=1,
        tree_method="hist",
    )),
])

rf_pipeline.fit(X_train, y_train)
xgb_pipeline.fit(X_train, y_train_bin)

rf_pred = rf_pipeline.predict(X_test)
xgb_pred = np.where(xgb_pipeline.predict(X_test) == 1, "Yes", "No")
rf_proba = rf_pipeline.predict_proba(X_test)[:, 1]
xgb_proba = xgb_pipeline.predict_proba(X_test)[:, 1]

joblib.dump(rf_pipeline, MODELS_DIR / "random_forest_churn.joblib")
joblib.dump(xgb_pipeline, MODELS_DIR / "xgboost_churn.joblib")

print("Random Forest and XGBoost fitted on", len(X_train), "train rows;", len(X_test), "test rows")
print("Saved random_forest_churn.joblib and xgboost_churn.joblib")
print("\nRandom Forest")
print("Accuracy:", round(accuracy_score(y_test, rf_pred), 3))
print(classification_report(y_test, rf_pred, digits=3))
print("XGBoost")
print("Accuracy:", round(accuracy_score(y_test, xgb_pred), 3))
print(classification_report(y_test, xgb_pred, digits=3))


#### Evaluation: compare Logistic Regression, Random Forest, XGBoost / ارزیابی مقایسه‌ای سه مدل

**English**

We report **accuracy**, **precision / recall / F1 for Yes (churn)**, **ROC-AUC**, and **average precision** on the test set. Higher ROC-AUC means better ranking of likely churners. F1 for Yes is the balance of catching churners vs false alarms.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

برای هر مدل **دقت**، **Precision / Recall / F1 کلاس Yes**، **ROC-AUC** و **Average Precision** روی آزمون گزارش می‌شود. ROC-AUC بالاتر یعنی رتبه‌بندی بهتر مشتریان در معرض ریزش.

</div>


In [ ]:
# Evaluation table for all three models
y_pred_lr = pipeline.predict(X_test)
y_proba_lr = pipeline.predict_proba(X_test)[:, 1]
y_true_bin = (y_test == "Yes").astype(int)

def model_eval_row(name, y_hat, y_prob):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_hat),
        "Precision_Yes": precision_score(y_test, y_hat, pos_label="Yes", zero_division=0),
        "Recall_Yes": recall_score(y_test, y_hat, pos_label="Yes", zero_division=0),
        "F1_Yes": f1_score(y_test, y_hat, pos_label="Yes", zero_division=0),
        "ROC_AUC": roc_auc_score(y_true_bin, y_prob),
        "Avg_Precision": average_precision_score(y_true_bin, y_prob),
    }

compare_df = pd.DataFrame([
    model_eval_row("Logistic Regression", y_pred_lr, y_proba_lr),
    model_eval_row("Random Forest", rf_pred, rf_proba),
    model_eval_row("XGBoost", xgb_pred, xgb_proba),
]).set_index("Model")
display(compare_df.round(3).sort_values("ROC_AUC", ascending=False))

# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, name, y_hat in zip(
    axes,
    ["Logistic Regression", "Random Forest", "XGBoost"],
    [y_pred_lr, rf_pred, xgb_pred],
):
    cm = confusion_matrix(y_test, y_hat, labels=["No", "Yes"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Pred No", "Pred Yes"], yticklabels=["Actual No", "Actual Yes"])
    ax.set_title(name)
fig.suptitle("Test confusion matrices", y=1.03)
fig.tight_layout()
plt.show()

# ROC + Precision-Recall
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for name, y_prob in [
    ("Logistic Regression", y_proba_lr),
    ("Random Forest", rf_proba),
    ("XGBoost", xgb_proba),
]:
    fpr, tpr, _ = roc_curve(y_true_bin, y_prob)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_true_bin, y_prob):.3f})")
    prec, rec, _ = precision_recall_curve(y_true_bin, y_prob)
    ap = average_precision_score(y_true_bin, y_prob)
    axes[1].plot(rec, prec, label=f"{name} (AP={ap:.3f})")
axes[0].plot([0, 1], [0, 1], "--", color="gray", label="Chance")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].set_title("ROC curve")
axes[0].legend(fontsize=8)
axes[1].axhline(y_true_bin.mean(), linestyle="--", color="gray", label="Prevalence")
axes[1].set_xlabel("Recall (Yes)")
axes[1].set_ylabel("Precision (Yes)")
axes[1].set_title("Precision-Recall curve")
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()


#### How to read these evaluation plots / چطور این نمودارهای ارزیابی را بخوانیم

**English**

- **Confusion matrix:** TN / FP on the first row (actual stay); FN / TP on the second row (actual churn).
- **ROC:** closer to the top-left is better. AUC 0.5 = random, 1.0 = perfect ranking.
- **Precision-Recall:** more useful with imbalanced churn (~26.5% Yes). Higher curve = better at finding churners without too many false alarms.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

- **ماتریس درهم‌ریختگی:** ردیف اول ماندن واقعی، ردیف دوم ریزش واقعی.
- **ROC:** نزدیک‌تر به گوشه بالا-چپ بهتر است.
- **Precision-Recall:** برای کلاس نامتوازن ریزش مفیدتر است.

</div>


## Predictions: Random Forest and XGBoost / پیش‌بینی با Random Forest و XGBoost

**English**

`predict` returns **No/Yes**. `predict_proba`[:, 1] is **P(churn = Yes)** — the score you would use to rank customers for retention. The tables below compare all three models on the same test customers.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

`predict` برچسب No/Yes می‌دهد. ستون احتمال یعنی **P(ریزش=Yes)** که برای رتبه‌بندی مشتریان برای نگهداری استفاده می‌شود.

</div>


In [ ]:
# Predictions on the first 10 test customers
X_test_subset = X_test.iloc[:10]
pred_subset = pd.DataFrame({
    "customerID": X_test_subset["customerID"].values if "customerID" in X_test_subset.columns else X_test_subset.index,
    "Actual": y_test.iloc[:10].values,
    "LogReg": pipeline.predict(X_test_subset),
    "RandomForest": rf_pipeline.predict(X_test_subset),
    "XGBoost": np.where(xgb_pipeline.predict(X_test_subset) == 1, "Yes", "No"),
    "LogReg_P(Yes)": pipeline.predict_proba(X_test_subset)[:, 1].round(3),
    "RF_P(Yes)": rf_pipeline.predict_proba(X_test_subset)[:, 1].round(3),
    "XGB_P(Yes)": xgb_pipeline.predict_proba(X_test_subset)[:, 1].round(3),
})
print("Predictions on first 10 test rows:")
display(pred_subset)

# Full test-set prediction table
tree_pred_df = X_test.copy()
tree_pred_df["Actual"] = y_test.values
tree_pred_df["Pred_LogReg"] = y_pred_lr
tree_pred_df["Pred_RF"] = rf_pred
tree_pred_df["Pred_XGB"] = xgb_pred
tree_pred_df["P_LogReg"] = y_proba_lr
tree_pred_df["P_RF"] = rf_proba
tree_pred_df["P_XGB"] = xgb_proba
tree_pred_df["RF_correct"] = tree_pred_df["Actual"] == tree_pred_df["Pred_RF"]
tree_pred_df["XGB_correct"] = tree_pred_df["Actual"] == tree_pred_df["Pred_XGB"]

print("\nRandom Forest accuracy (mean of RF_correct):", tree_pred_df["RF_correct"].mean().round(3))
print("XGBoost accuracy (mean of XGB_correct):", tree_pred_df["XGB_correct"].mean().round(3))
print("\nHighest-risk test customers by XGBoost P(churn):")
top_cols = [c for c in ["customerID", "Contract", "InternetService", "tenure", "MonthlyCharges",
                        "Actual", "Pred_RF", "Pred_XGB", "P_RF", "P_XGB"] if c in tree_pred_df.columns]
display(tree_pred_df.sort_values("P_XGB", ascending=False)[top_cols].head(10))


#### Feature importance (trees) and how to use the predictions / اهمیت ویژگی‌ها و استفاده از پیش‌بینی

**English**

Tree models report which one-hot / numeric features they split on most. Month-to-month contract and tenure usually rank high for this dataset. Use **P(Yes)** to build a retention list: contact the highest probabilities first.

---

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 1.05em;">

**فارسی**

مدل‌های درختی نشان می‌دهند کدام ویژگی‌ها بیشتر در تقسیم درخت استفاده شده‌اند. معمولاً قرارداد ماه‌به‌ماه و tenure مهم‌اند. برای تماس نگهداری، مشتریان با بالاترین **P(Yes)** را اول انتخاب کنید.

</div>


In [ ]:
# Feature importance from Random Forest and XGBoost
ohe = rf_pipeline.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]
num_names = list(numerical_cols)
cat_names = list(ohe.get_feature_names_out(categorical_cols))
feat_names = num_names + cat_names

rf_imp = pd.Series(rf_pipeline.named_steps["classifier"].feature_importances_, index=feat_names).sort_values(ascending=False)
xgb_imp = pd.Series(xgb_pipeline.named_steps["classifier"].feature_importances_, index=feat_names).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
rf_imp.head(12).iloc[::-1].plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].set_title("Random Forest · top 12 features")
axes[0].set_xlabel("Importance")
xgb_imp.head(12).iloc[::-1].plot(kind="barh", ax=axes[1], color="teal")
axes[1].set_title("XGBoost · top 12 features")
axes[1].set_xlabel("Importance")
fig.tight_layout()
plt.show()

print("Random Forest top feature:", rf_imp.index[0])
print("XGBoost top feature:", xgb_imp.index[0])
print("\nTo score new customers:")
print("  rf_pipeline.predict(new_X)           # No / Yes")
print("  rf_pipeline.predict_proba(new_X)[:, 1]  # P(churn)")
print("  xgb_pipeline.predict_proba(new_X)[:, 1]  # P(churn); labels are 0/1 internally")
